# 02 2 つの表を結合して、人口あたりに直す

**この課題で体験するデータ工学的な難しさ**
1. 2 つの表を「都道府県名」でつなぐとき、表記が少し違うだけでつながらない（「東京」と「東京都」、末尾の空白、旧字体）。
2. つながらなかった行は黙って消える。だから **結合のあとに行数を数える**。
3. 「病院数が多い県」と「人口あたり病院数が多い県」は別物。比べるときは分母を揃える。

**やること**：上から順に実行。「★」の行だけ書き換えてよい。

## 1. 2 つの表を読む

In [ ]:
import pandas as pd
jinko = pd.read_csv("../data/todofuken_jinko.csv", dtype={"都道府県コード": str})
byoin = pd.read_csv("../data/todofuken_byoin.csv")
print(len(jinko), "行と", len(byoin), "行")
byoin.head(8)

In [ ]:
結合1 = jinko.merge(byoin, left_on="都道府県", right_on="都道府県名", how="inner")
print("結合できた行数:", len(結合1), "/ 47")

In [ ]:
結合1 = jinko.merge(byoin, left_on="都道府県", right_on="都道府県名", how="inner")
print("結合できた行数:", len(结合1) if False else len(結合1), "/ 47")

## 3. 結合できなかった行はどれか（必ず確かめる）
`how="left"` にすると、相手が見つからない行も残り、病院数が空（NaN）になる。

In [ ]:
確認 = jinko.merge(byoin, left_on="都道府県", right_on="都道府県名", how="left")
確認[確認["病院数"].isna()][["都道府県コード", "都道府県"]]

## 4. 相手の表の表記を見る
空白や「都・府・県」の有無、旧字体が混ざっていないか。`repr` で見ると空白が見える。

In [ ]:
for 名前 in byoin["都道府県名"]:
    if 名前 not in set(jinko["都道府県"]):
        print(repr(名前))

## 5. キーを正規化する
前後の空白（全角も）を取り、「都・府・県」を補い、旧字体を直す。正規化は **新しい列** に入れて、元の列は残す。

In [ ]:
def 正規化(s):
    s = str(s).replace("\u3000", " ").strip()
    s = s.replace("沖繩", "沖縄")
    if s == "北海道":
        return s
    if s in ("東京",):
        return "東京都"
    if s in ("大阪", "京都"):
        return s + "府"
    if not s.endswith(("都", "府", "県")):
        return s + "県"
    return s

byoin["都道府県_正規化"] = byoin["都道府県名"].map(正規化)
結合2 = jinko.merge(byoin, left_on="都道府県", right_on="都道府県_正規化", how="left")
print("病院数が空の行数:", 結合2["病院数"].isna().sum(), "（0 になれば成功）")

## 6. 人口 10 万人あたりに直す

In [ ]:
結合2["病院数_10万人あたり"] = 結合2["病院数"] / 結合2["人口_2020"] * 100000
列 = ["都道府県", "人口_2020", "病院数", "病院数_10万人あたり"]
print("--- 病院数そのもの 上位 5")
display(結合2.sort_values("病院数", ascending=False)[列].head(5))
print("--- 人口 10 万人あたり 上位 5")
display(結合2.sort_values("病院数_10万人あたり", ascending=False)[列].head(5))
print("--- 人口 10 万人あたり 下位 5")
display(結合2.sort_values("病院数_10万人あたり", ascending=True)[列].head(5))

## 7. 確かめること（提出用に 3 行で書く）
- 正規化しないと何県が消えたか。消えたことに気づく方法は何だったか。
- 「病院数 上位 5」と「10 万人あたり 上位 5」で顔ぶれはどう変わったか。どちらが「病院が身近な県」を表すか。
- ★ 分母を `面積_km2` に変えて（`* 100` にして 100 km² あたり）実行すると、順位はどう変わるか。